### 🧠 Deadline Manager Agent – EY AI Challenge

Modular notebook: OCR, date parsing, working-days, LLM agent para prazos legais e integração opcional de calendário.

In [21]:
# DEPENDENCIES: Instale as bibliotecas necessárias via pip.
!pip install --upgrade pytesseract PyPDF2 pillow dateparser python-dateutil holidays transformers huggingface_hub[hf_xet]
!pip install --upgrade pytesseract PyPDF2 pillow dateparser python-dateutil holidays transformers huggingface_hub[hf_xet]

In [ ]:
# IMPORTS: Apenas o que ainda não foi importado em outras células
from datetime import datetime, timedelta
from dateparser.search import search_dates
from dateutil.relativedelta import relativedelta
import holidays
import dateparser
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from docx import Document


### 🖼️ OCR & PDF Extraction
Functions to read text in images (Tesseract) and PDFs.

In [22]:
def extract_text_from_image(path):
    """Base da extração de texto a partir de uma imagem (em português)."""
    return pytesseract.image_to_string(Image.open(path), lang='por')

def extract_text_from_pdf(path):
    """Base da extração de texto de todas as páginas de um PDF."""
    rdr = PdfReader(path)
    return "\n".join(page.extract_text() or "" for page in rdr.pages)

In [13]:
import os

print(os.getcwd())
print("Existe diretório Data?", os.path.isdir('Data'))

c:\Users\ferol\Documents\GitHub\Auto-Calendar-Agent
Existe diretório Data? False


In [23]:
# Se estiver em ambiente Windows, instale o Tesseract manualmente ou garanta que está no PATH.
# Em notebooks Google Colab/Linux, use: !apt-get update && apt-get install -y tesseract-ocr
# Aqui, apenas um aviso para o usuário:
print("Tesseract não está instalado ou não está no PATH. Veja o arquivo README para mais informações.")

import glob
# Busca arquivos .jpeg e .jpg
img_files = glob.glob('Data/*.jpeg') + glob.glob('Data/*.jpg')
print(img_files)  # Verifique se encontra arquivos

for img_path in img_files:
    texto_extraido = extract_text_from_image(img_path)
    print(f"Arquivo: {img_path}\nTexto extraído:\n{texto_extraido}\n{'-'*40}\n")

# Busca arquivos .pdf
pdf_files = glob.glob('Data/*.pdf')
print(pdf_files)  # Verifique se encontra arquivos

for pdf_path in pdf_files:
    texto_extraido_pdf = extract_text_from_pdf(pdf_path)
    print(f"Arquivo: {pdf_path}\nTexto extraído:\n{texto_extraido_pdf}\n{'-'*40}\n")


Tesseract não está instalado ou não está no PATH. Veja o arquivo README para mais informações.
['Data\\Post-it To Do DMR ABC Co.jpeg', 'Data\\Post-it To Do DMR Maio 2025 ACE.jpeg', 'Data\\Post-it To Do DP IVA ACE.jpeg', 'Data\\Post-it To Do DP IVA Junho 2025 ABC.Co.jpeg', 'Data\\Post-it To Do IES ACE.jpeg', 'Data\\Post-it To Do Modelo 30 ACE.jpeg', 'Data\\Post-it To Do Retencao na Fonte Modelo 10 ABC.Co.jpeg', 'Data\\Post-it To Do SAF-T Acceta.jpeg', 'Data\\Post-it To Do SAF-T Junho 2025 ACE.jpeg', 'Data\\Post-it To Do SAF-T Maio 2025 ABC.Co.jpeg']


TesseractNotFoundError: tesseract is not installed or it's not in your PATH. See README file for more information.

### 🧠 Data extraction (NLU)
Extract the first future date from a free text like `dateparser.search.search_dates`.

In [ ]:
def infer_deadline(text, base_date=None):
    """Base da identificação de uam data a partir de uma imagem."""
    base = base_date or datetime.now()
    res = search_dates(
        text,
        languages=['pt','en'],
        settings={
            'PREFER_DATES_FROM':'future',
            'RELATIVE_BASE':base,
            'DATE_ORDER':'DMY'
        }
    )
    return res[0][1] if res else None

In [ ]:
# Extração de texto e data para cada imagem em img_files
for img_path in img_files:
    texto = extract_text_from_image(img_path)
    data_inferida_imagem = infer_deadline(texto)
    print(f"Arquivo: {img_path}\nTexto extraído:\n{texto}\nData inferida: {data_inferida}\n{'-'*40}\n")

# Extração de texto e data para cada PDF em pdf_files
for pdf_path in pdf_files:
    texto_pdf = extract_text_from_pdf(pdf_path)
    data_inferida_pdf = infer_deadline(texto_pdf)
    print(f"Arquivo: {pdf_path}\nTexto extraído:\n{texto_pdf}\nData inferida: {data_inferida_pdf}\n{'-'*40}\n")

### 📅 Work days calculation (PT)
Add work days to a date, excluding weekends and Portuguese holidays.

In [ ]:
def add_working_days(start_date, days):
    """Base de unção auxiliar para somar dias úteis a uma data, gerir férias judiciais, etc."""
    pt_hols = holidays.Portugal()
    curr = start_date
    added = 0
    while added < days:
        curr += relativedelta(days=1)
        if curr.weekday() < 5 and curr not in pt_hols:
            added += 1
    return curr

In [ ]:
# Para cada imagem em img_files, extrai texto, infere data e soma 5 dias úteis
for img_path in img_files:
    texto = extract_text_from_image(img_path)
    data_inferida = infer_deadline(texto)
    if data_inferida:
        prazo_final = add_working_days(data_inferida, 5)
        print(f"Arquivo: {img_path}\nData inferida: {data_inferida}\nPrazo final (+5 dias úteis): {prazo_final}\n{'-'*40}\n")
    else:
        print(f"Arquivo: {img_path}\nData não encontrada.\n{'-'*40}\n")

In [ ]:

# Exemplo de uso:
# resultado = process_deadline_from_image_or_text(img_path, is_image=True)
# print(resultado)

In [ ]:
import glob
from docx import Document

# Processa todos os arquivos de imagem, PDF e Word usando process_deadline_from_image_or_text

# Arquivos de imagem e PDF já definidos
# img_files, pdf_files já definidos acima

# Busca arquivos .docx e .doc
word_files = glob.glob('Data/*.docx') + glob.glob('Data/*.doc')

def extract_text_from_word(path):
    """Extrai texto de arquivos Word (.docx)."""
    doc = Document(path)
    return "\n".join([p.text for p in doc.paragraphs])

# Processa imagens
for img_path in img_files:
    resultado = process_deadline_from_image_or_text(img_path, is_image=True)
    print(f"Imagem: {img_path}\nResultado: {resultado}\n{'-'*40}")

# Processa PDFs
for pdf_path in pdf_files:
    resultado = process_deadline_from_image_or_text(pdf_path, is_image=False)
    print(f"PDF: {pdf_path}\nResultado: {resultado}\n{'-'*40}")

# Processa Word
for word_path in word_files:
    texto_word = extract_text_from_word(word_path)
    resultado = process_deadline_from_image_or_text(texto_word, is_image=False)
    print(f"Word: {word_path}\nResultado: {resultado}\n{'-'*40}")


### 🤖 Deadline Agent (LLM Free)
One type of open-source model (Flan-T5 small) to apply the following rules:
- Modelo 22: up to 31/jul
- IES: 15/apr (current and next year)
- Others: infer via NLP

In [ ]:
# Implementation using simple LLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model     = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

def llm_generate(prompt: str, max_length: int = 256) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").input_ids
    outs = model.generate(
        inputs, num_beams=4, early_stopping=True, max_length=max_length
    )
    return tokenizer.decode(outs[0], skip_special_tokens=True)

def agent_process(text, reference_date=None):
    """ Base de um Agente que infere deadlines aplicando regras legais ou simplesmente Língua Natural. Retorna a data em dicionário apto para JSON {'deadline': datetime} ou {'error':...}."""

    ref = reference_date or datetime.now()
    
    prompt = f"""
You are a Portuguese legal deadline assistant. Determine the deadline for the request below using these rules:
- "Modelo 22": due by {ref.year}-07-31
- "IES": due by {ref.year}-04-15 if before, else {ref.year+1}-04-15
- Otherwise infer via natural language (e.g. "5 working days from now").
Reference date: {ref.strftime('%Y-%m-%d')}
Input: "{text}"
Return ONLY a JSON object with key "deadline" (ISO8601 date string).
"""
    
    raw = llm_generate(prompt)
    
    try:
        obj = json.loads(raw)
        d = dateparser.parse(obj['deadline'])
        return {'deadline': d}
    except Exception as e:
        return {'error': f'LLM parse error: {e} | raw: {raw}'}

In [ ]:
# Implementation using Gemini LLM

def config_llm_gemini(temperature:int):
  '''LLM api calling using Gemini  '''
  # Steps for students:
  # - Go to https://aistudio.google.com/app/apikey and generate your Gemini API key.
  # - Add the necessary packages to your requirements.txt:
  #    langchain
  #    langchain-google-genai
  # - Run the following command to install them:
  #     !pip install -r requirements.txt
  # - Follow the official integration guide for LangChain + Google Generative AI:
  #     https://python.langchain.com/docs/integrations/chat/google_generative_ai/
  # Pay attention to the request limits of the chosen model.
  return "llm" #Should return the LLM response

### 🔗 Calendar integration (Opcional)
Function to create events in external calendar tool

In [ ]:
# def create_calendar_event(summary, start, end, timezone='UTC'):
#     pass  # implementar conforme API desejada

### 🧪 Use case examples

In [ ]:
# Exemplo OCR:
# img_text = extract_text_from_image('scan.png')
# print(infer_deadline(img_text))

# Exemplo agente:
# print(agent_process('Entregar Modelo 22'))
# print(agent_process('Enviar IES até dia 15 de abril'))

# Working days:
# base = datetime(2025,5,27)
# print(add_working_days(base,5))